# py4conjoint — 全公開API デモ

このノートブックは `py4conjoint` の **全ユーザー向け関数・属性・メソッド** を
合成データで動作確認するためのものです。

| セクション | 内容 |
|:---:|------|
| 0 | セットアップ |
| 1 | 合成データの準備 |
| 2 | `pc.forms_to_conjoint_data()` |
| 3 | `pc.auto_reference_levels()` |
| 4 | `pc.encode()` |
| 5 | `pc.fit()` |
| 6 | `ConjointResult` のフィールド |
| 7 | `result.summary()` |
| 8 | `result.warnings()` |
| 9 | `result.importance()` |
| 10 | `result.wtp()` |
| 11 | `result.unit_rating_money()` |
| 12 | `result.market_share()` |
| 13 | 可視化（`plot_importance / plot_partworth / plot_wtp`） |

## 0. セットアップ

In [ ]:
import numpy as np
import pandas as pd
import py4conjoint as pc

print(f"py4conjoint version: {pc.__version__}")

## 1. 合成データの準備

コンジョイント分析では **カード（プロファイル）** を回答者に提示し、各カードを評価してもらいます。
ここでは価格・OS・カメラの 3 属性・各 2 水準（4 カード）の設計を使用します。

| カードID | price（万円） | os | camera |
|----------|:-----------:|:-----:|:------:|
| P1 | 6 | android | 標準 |
| P2 | 10 | apple | 標準 |
| P3 | 6 | apple | 高性能 |
| P4 | 10 | android | 高性能 |

`forms_to_conjoint_data()` の出力と同じ **long 形式**（1 行 = 1 人 × 1 カード）を生成します。

想定する真の係数： `price_0 = +1.25`、`os_0 = +0.8`、`camera_0 = +0.6`

In [ ]:
cards = pd.DataFrame(
    {
        "price":  [6, 10, 6, 10],
        "os":     ["android", "apple", "apple", "android"],
        "camera": ["標準", "標準", "高性能", "高性能"],
    },
    index=["P1", "P2", "P3", "P4"],
)

rng = np.random.default_rng(42)
rows = []
for resp_id in range(1, 31):          # 30 人
    for card_id, (_, row) in zip(cards.index, cards.iterrows()):
        utility = (
            -1.25 * (1 if row["price"] == 10 else -1)
            + 0.80 * (1 if row["os"] == "apple" else -1)
            + 0.60 * (1 if row["camera"] == "高性能" else -1)
        )
        rating = int(round(utility * 2 + 5 + rng.normal(0, 0.3)))
        rating = max(1, min(7, rating))
        rows.append(
            {
                "回答者ID": resp_id,
                "カードID": card_id,
                "rating": rating,
                "price": row["price"],
                "os": row["os"],
                "camera": row["camera"],
            }
        )
df = pd.DataFrame(rows)

print(f"行数: {len(df)} = {df['回答者ID'].nunique()} 人 × {len(cards)} カード")
df.head(8)

## 2. `pc.forms_to_conjoint_data()` — アンケートファイルの読み込み

Microsoft Forms（.xlsx）または Google Forms（.csv）の回答ファイルを
**long 形式 DataFrame** に変換する関数です。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `responses_file` | str | — | アンケートファイルのパス（.xlsx / .csv） |
| `n_cards` | int | — | 提示したカード枚数 |
| `attributes` | DataFrame or list of dict | — | カード設計（行：カード、列：属性） |
| `forms` | str | `"microsoft"` | `"microsoft"` または `"google"` |
| `respondent_cols` | dict | `None` | 回答者属性として残したい列の対応辞書 |
| `card_id_prefix` | str | `"P"` | カードIDの接頭辞（P1, P2, ...） |
| `rating_colname` | str | `"rating"` | 出力 DataFrame の評点列名 |
| `respondent_id_colname` | str | `"回答者ID"` | 出力 DataFrame の回答者 ID 列名 |
| `card_id_colname` | str | `"カードID"` | 出力 DataFrame のカード ID 列名 |
| `out_csv` | str | `None` | 変換後 DataFrame を CSV で保存するパス |

**返り値**: long 形式 DataFrame（列：回答者ID, カードID, rating, 回答者属性, カード属性）

In [ ]:
# このセルは実際のアンケートファイルが必要なため、そのまま実行しても
# FileNotFoundError になります。実際に使う場合は responses_file を
# 実際のファイルパスに書き換えてから実行してください。

# df_from_forms = pc.forms_to_conjoint_data(
#     responses_file="responses.xlsx",   # Microsoft Forms の場合
#     n_cards=4,
#     attributes=cards,                  # DataFrame 形式（推奨）
#     forms="microsoft",                 # "google" も可
#     respondent_cols={"性別": "gender", "学年": "grade"},  # 回答者属性（任意）
#     card_id_prefix="P",
#     rating_colname="rating",
#     respondent_id_colname="回答者ID",
#     card_id_colname="カードID",
#     out_csv="conjoint_data.csv",       # CSV 保存（任意）
# )

# attributes は辞書のリスト形式でも指定できる
# df_from_forms = pc.forms_to_conjoint_data(
#     responses_file="responses.xlsx",
#     n_cards=4,
#     attributes=[
#         {"price":  [6, 10, 6, 10]},
#         {"os":     ["android", "apple", "apple", "android"]},
#         {"camera": ["標準", "標準", "高性能", "高性能"]},
#     ],
# )

print("forms_to_conjoint_data() の使い方はコメントを参照してください。")
print("このノートブックでは合成データ df をそのまま使います。")

## 3. `pc.auto_reference_levels()` — 基準水準の自動推測

各属性の基準水準（`encode()` で `-1` になる水準）を自動で推測する補助関数です。

**判定ルール**
- 数値列（価格相当）: **最大値** を基準
- カテゴリ列: 文字列で辞書順ソートして **先頭** を基準

⚠️ 推論ベースなので、内容を確認してから `encode()` に渡してください。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `df` | DataFrame | — | 属性列を含む DataFrame |
| `attribute_columns` | list of str | — | 基準水準を推測したい属性名リスト |
| `price_columns` | list of str | `["price"]` | 価格相当の列名（最大値を基準にする） |

**返り値**: `dict` — そのまま `encode()` の `reference_levels` に渡せる

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    auto_refs = pc.auto_reference_levels(
        df,
        attribute_columns=["price", "os", "camera"],
        price_columns=["price"],   # 価格列を明示（最大値を基準にする）
    )

print("自動推測された基準水準:")
for attr, ref in auto_refs.items():
    print(f"  {attr}: {ref!r}")

if caught:
    print(f"\n[警告メッセージ]\n{caught[0].message}")

## 4. `pc.encode()` — 効果コーディング（-1 / +1）

属性列を **効果コーディング（±1）** に自動変換する関数です。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `df` | DataFrame | — | long 形式のデータ |
| `reference_levels` | dict | — | `{"属性名": 基準水準}` の辞書。基準水準が `-1` になる |
| `binary_suffix_map` | dict | `None` | 2 水準属性の列名サフィックスを手動指定 |
| `drop_original` | bool | `False` | 元の属性列を削除するか |
| `inplace` | bool | `False` | 入力 df を直接書き換えるか |

**返り値の列名ルール**
- 2 水準 → `{属性名}_0` が 1 列追加
- 3 水準以上 → `{属性名}_0`, `{属性名}_1`, … が K-1 列追加
- `binary_suffix_map` 指定時 → `{属性名}_{指定サフィックス}`

In [ ]:
# 基本的な使い方
df_coded = pc.encode(
    df,
    reference_levels={
        "price":  10,        # 高い方を基準 → 安い方(6万)が +1
        "os":     "android",
        "camera": "標準",
    },
)

added_cols = [c for c in df_coded.columns if c not in df.columns]
print("追加された符号化列:", added_cols)
df_coded[["回答者ID", "カードID", "rating", "price_0", "os_0", "camera_0"]].head(8)

In [ ]:
# binary_suffix_map: 列名のサフィックスを手動指定
df_named = pc.encode(
    df,
    reference_levels={"price": 10, "os": "android", "camera": "標準"},
    binary_suffix_map={"price": "low", "os": "apple", "camera": "high"},
)
print("binary_suffix_map 指定時の列名:",
      [c for c in df_named.columns if c not in df.columns])

In [ ]:
# drop_original=True: 元の属性列を削除
df_dropped = pc.encode(
    df,
    reference_levels={"price": 10, "os": "android", "camera": "標準"},
    drop_original=True,
)
print("drop_original=True 後の列:", list(df_dropped.columns))

In [ ]:
# inplace=True: 入力 df を直接書き換える（コピーを返さない）
df_copy = df.copy()
result_inplace = pc.encode(
    df_copy,
    reference_levels={"price": 10, "os": "android", "camera": "標準"},
    inplace=True,
)
print("inplace=True: 戻り値は元の df と同一オブジェクト:",
      result_inplace is df_copy)
print("追加列:", [c for c in df_copy.columns if c not in df.columns])

In [ ]:
# 3 水準以上の属性：K-1 列が生成される
df_multi = pd.DataFrame(
    {
        "rating": [5, 3, 7, 4, 6, 2],
        "color":  ["赤", "青", "緑", "赤", "青", "緑"],
    }
)
df_multi_coded = pc.encode(
    df_multi,
    reference_levels={"color": "赤"},  # 3 水準 → color_0（青）, color_1（緑）
)
print("3 水準属性の符号化（基準=赤）:")
print(df_multi_coded)

## 5. `pc.fit()` — OLS 回帰

符号化済み DataFrame に OLS 回帰を適用し、`ConjointResult` を返す関数です。
欠損値の除外、日本語列名の自動エイリアス処理、落とし穴チェックを自動実行します。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `df` | DataFrame | — | `encode()` 済みの DataFrame |
| `rating` | str | `"rating"` | 評点（被説明変数）の列名 |
| `price_col` | str | `"price"` | 価格列名。WTP 計算で使用 |
| `encoded_columns` | list of str | `None` | 説明変数リスト（省略時は自動検出） |
| `reference_levels` | dict | `None` | 基準水準辞書（`encode()` 経由なら自動取得） |
| `formula` | str | `None` | statsmodels 用の回帰式を直接指定 |

**返り値**: `ConjointResult` オブジェクト

In [ ]:
# 基本的な使い方（encode() 後は reference_levels が df.attrs に保存されている）
result = pc.fit(df_coded, price_col="price")
print("fit() 完了")

In [ ]:
# encoded_columns を明示的に指定する例
result_explicit = pc.fit(
    df_coded,
    encoded_columns=["price_0", "os_0", "camera_0"],
    price_col="price",
)
print("encoded_columns 明示指定でも同じ係数が得られる:")
print(result_explicit.params)

## 6. `ConjointResult` のフィールド

### 6-A. プロパティ（頻繁に使う）

| プロパティ | 型 | 説明 |
|-----------|-----|------|
| `params` | `pd.Series` | 推定係数（切片含む） |
| `rsquared` | `float` | 決定係数 R² |
| `n_obs` | `int` | 分析に使用した観測数 |
| `intercept` | `float` | 切片 b₀（全水準平均の効用） |

### 6-B. データフィールド（中級者向け）

| フィールド | 型 | 説明 |
|-----------|-----|------|
| `rating` | `str` | 評点列名 |
| `encoded_columns` | `list` | 説明変数（符号化列）のリスト |
| `reference_levels` | `dict` | `encode()` に渡された基準水準辞書 |
| `price_col` | `str` | 価格列名 |
| `n_dropped` | `int` | 欠損除外された行数 |
| `alias_map` | `dict` | 符号化列名の内部エイリアスマップ（`formula=None` 時に生成、内部処理用） |
| `model_result` | `RegressionResults` | statsmodels の生の推定結果 |

In [ ]:
# プロパティ
print("result.params")
print(result.params)
print()
print(f"result.rsquared  = {result.rsquared:.4f}")
print(f"result.n_obs     = {result.n_obs}")
print(f"result.intercept = {result.intercept:.4f}")

In [ ]:
# データフィールド
print(f"result.rating            = {result.rating!r}")
print(f"result.encoded_columns   = {result.encoded_columns}")
print(f"result.reference_levels  = {result.reference_levels}")
print(f"result.price_col         = {result.price_col!r}")
print(f"result.n_dropped         = {result.n_dropped}")
print(f"result.alias_map         = {result.alias_map}")

In [ ]:
# model_result: statsmodels の RegressionResults をそのまま利用
print(result.model_result.summary())

## 7. `result.summary()` — 和文サマリー

係数表・R²・落とし穴チェック結果を **日本語** でまとめて表示します。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `slim` | bool | `True` | `True` → コンパクトな係数表、`False` → statsmodels 標準の詳細表 |

**返り値**: `str` — 重大度「大」の落とし穴のみ表示（「中」「小」は `warnings()` で確認）

In [ ]:
# slim=True（デフォルト）
print(result.summary())

In [ ]:
# slim=False: statsmodels の詳細な統計表（英語）を表示
print(result.summary(slim=False))

## 8. `result.warnings()` — 落とし穴チェック

回帰分析で自動検出された **落とし穴（診断警告）** の一覧を返します。

**自動検出される警告の種類**

| カテゴリ | 重大度 | 検出タイミング | 内容 |
|----------|:------:|:-----------:|------|
| `r2_low` | 大 | `fit()` 直後 | R² < 0.20 |
| `few_respondents` | 大/中 | `fit()` 直後 | 回答者数が少ない（1人 → 大、2〜4人 → 中、5人以上は警告なし） |
| `price_sign_negative` | 中 | `fit()` 直後 | 価格係数の符号が逆（価格↑で評点↑） |
| `price_insignificant` | 中 | `wtp()` 呼出時 | 価格係数の p 値 ≥ 0.10 |
| `wtp_extrapolation` | 大/中 | `wtp()` 呼出時 | \|WTP\| > 価格レンジ × 2 |

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `severity` | str / list | `None` | 重大度でフィルタ（`"大"`/`"中"`/`"小"`） |
| `category` | str / list | `None` | カテゴリ名でフィルタ |
| `as_dataframe` | bool | `True` | `True` → DataFrame、`False` → `Diagnostic` リスト |

**返り値**: `pd.DataFrame`（列: severity, category, message, recommendation）または `list[Diagnostic]`

In [ ]:
# --- 正常なデータ（30 人分）では警告なし ---

# wtp() を先に呼んで価格関連の警告も生成させる
_ = result.wtp()

print("【すべての警告】")
all_warnings = result.warnings()
if len(all_warnings) == 0:
    print("  警告なし（データ品質は良好）")
else:
    print(all_warnings.to_string(index=False))

In [ ]:
# --- 問題のあるデータ（3 人分・純ランダム評点）で警告をデモ ---

rng_bad = np.random.default_rng(99)
rows_bad = []
for resp_id in range(1, 4):                 # 3 人のみ
    for card_id, (_, row) in zip(cards.index, cards.iterrows()):
        rows_bad.append(
            {
                "回答者ID": resp_id,
                "カードID": card_id,
                "rating":  int(rng_bad.integers(1, 8)),  # ランダム評点
                "price":   row["price"],
                "os":      row["os"],
                "camera":  row["camera"],
            }
        )
df_bad = pd.DataFrame(rows_bad)
df_bad_coded = pc.encode(
    df_bad,
    reference_levels={"price": 10, "os": "android", "camera": "標準"},
)
result_bad = pc.fit(df_bad_coded, price_col="price")
_ = result_bad.wtp()

print("【問題データの警告一覧】")
bad_warnings = result_bad.warnings()
if len(bad_warnings) == 0:
    print("  警告なし")
else:
    for _, row in bad_warnings.iterrows():
        print(f"  [{row['severity']}] {row['category']}")
        print(f"    {row['message']}")

In [ ]:
# severity でフィルタ
print("【重大度 '大' のみ】")
print(result_bad.warnings(severity="大"))

# category でフィルタ
print("\n【カテゴリ 'few_respondents' のみ】")
print(result_bad.warnings(category="few_respondents"))

# as_dataframe=False で Diagnostic オブジェクトのリストとして取得
print("\n【as_dataframe=False: Diagnostic リスト】")
diag_list = result_bad.warnings(as_dataframe=False)
for d in diag_list:
    print(f"  severity={d.severity!r}, category={d.category!r}")
    print(f"    message: {d.message[:60]}...")
    print(f"    recommendation: {d.recommendation[:60]}...")

## 9. `result.importance()` — 相対重要度

各属性の **効用範囲（最大 − 最小）の比率** として相対重要度を計算します。

**計算式**
$$\text{importance}_i = \frac{\text{range}_i}{\sum_j \text{range}_j} \times 100$$

2 水準の場合の効用範囲 = `2 × |係数|`（水準が -1 と +1 なので差は 2 倍）

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `as_percent` | bool | `True` | `True` → 合計 100%、`False` → 合計 1.0 の比率 |

**返り値**: `pd.DataFrame`（インデックス: 属性名、列: `range`, `importance`）

In [ ]:
print("【as_percent=True（デフォルト）— 合計 100%】")
imp_pct = result.importance(as_percent=True)
print(imp_pct)
print(f"  合計: {imp_pct['importance'].sum():.1f}%")

print()
print("【as_percent=False — 合計 1.0 の比率】")
imp_ratio = result.importance(as_percent=False)
print(imp_ratio)
print(f"  合計: {imp_ratio['importance'].sum():.4f}")

## 10. `result.wtp()` — WTP（支払意思額）

各非価格属性の **WTP（Willingness to Pay）** を計算します。
「基準水準から非基準水準に変えるとき、回答者が最大いくら追加で支払うか」を金額で表します。

**計算式**
$$\text{WTP}_{\text{attr}} = \underbrace{\frac{\text{price\_range}}{b_{\text{price}}}}_{\text{wtp\_price\_factor}} \times b_{\text{attr}}$$

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `price_col` | str | `None` | 価格列名（省略時は `fit()` の設定値） |

**返り値**: `pd.DataFrame`（インデックス: 符号化列名、列: `coef`, `wtp`）  
価格列と同じ単位（万円・千円・円など）で返されます。

In [ ]:
wtp_df = result.wtp()
print(wtp_df)

# attrs に保存されたメタ情報も確認
print(f"\n  price_range      = {wtp_df.attrs['price_range']}")
print(f"  wtp_price_factor = {wtp_df.attrs['wtp_price_factor']:.4f}")
print(f"  p_price          = {wtp_df.attrs['p_price']:.4f}")

## 11. `result.unit_rating_money()` — 評点 1 点の金額換算

評点 1 ポイントが何円（または何万円）に相当するかを返します。

**計算式**
$$\text{unit\_rating\_money} = \frac{\text{price\_range}}{|b_{\text{price}}| \times 2}$$

効果コーディングで価格変数の範囲が `-1` 〜 `+1` の 2 単位であることに由来します。  
WTP の `wtp_price_factor`（= price_range / b_price）とは **異なる値** です（2 倍の差）。

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `price_col` | str | `None` | 価格列名（省略時は `fit()` の設定値） |

**返り値**: `float` — 価格列と同じ単位

In [ ]:
unit = result.unit_rating_money()
print(f"評点 1 点 = {unit:.4f} 万円")

# 手計算で確認
b_price = result.params["price_0"]
price_range = 10 - 6
print(f"検算: {price_range} / (|{b_price:.4f}| × 2) = {price_range / abs(b_price * 2):.4f}")

# wtp_price_factor との関係
print(f"\nwtp_price_factor = {wtp_df.attrs['wtp_price_factor']:.4f}")
print(f"unit_rating_money = wtp_price_factor / 2 = {wtp_df.attrs['wtp_price_factor'] / 2:.4f}")

## 12. `result.market_share()` — 市場シェア予測

複数製品の効用を推定し、市場シェアを予測します。

**計算方法**

| `method` | 説明 |
|----------|------|
| `"logit"`（デフォルト） | ソフトマックス型。`share_i = exp(u_i) / Σ exp(u_j)` |
| `"share_of_preference"` | `"logit"` の別名（同じ計算） |
| `"max"` | 最大効用ルール。最も効用が高い製品にシェア 1.0 |

**主要引数**

| 引数 | 型 | デフォルト | 説明 |
|------|----|-----------|------|
| `products` | DataFrame | — | 製品 × 符号化列の DataFrame |
| `method` | str | `"logit"` | `"logit"` / `"share_of_preference"` または `"max"` |

**返り値**: `pd.Series`（インデックス: 製品名、値: シェア 0〜1）  
合計は必ず 1.0 になります。

In [ ]:
products = pd.DataFrame(
    {
        "price_0":  [ 1, -1,  1],   # 6万円=1, 10万円=-1
        "os_0":     [ 1,  1, -1],   # apple=1, android=-1
        "camera_0": [ 1,  1, -1],   # 高性能=1, 標準=-1
    },
    index=["製品A（6万・apple・高性能）", "製品B（10万・apple・高性能）", "製品C（6万・android・標準）"],
)

print('【method="logit"（デフォルト）】')
share_logit = result.market_share(products, method="logit")
print(share_logit)
print(f"  合計: {share_logit.sum():.6f}")

print()
print('【method="max"（最大効用ルール）】')
share_max = result.market_share(products, method="max")
print(share_max)
print(f"  合計: {share_max.sum():.6f}")

## 13. 可視化

3 つの可視化関数はいずれも **`result.plot_*()`** （ConjointResult メソッド）と
**`pc.plot_*()`**（モジュールレベル関数）の 2 通りで呼び出せます。

すべて `matplotlib.axes.Axes` を返すので `ax.set_title()` 等で後から調整できます。

| 関数/メソッド | 主な引数 | 説明 |
|---|---|---|
| `plot_importance()` | `sort`, `show_values`, `color` | 相対重要度の水平棒グラフ |
| `plot_partworth()` | `show_zero_line` | 各水準の部分効用グラフ（属性ごとに色分け） |
| `plot_wtp()` | `sort`, `show_values`, `price_unit` | WTP の水平棒グラフ |

In [ ]:
# plot_importance — result メソッドとして呼ぶ
ax = result.plot_importance(title="属性の相対重要度", show_values=True, sort=True)
ax.set_xlabel("重要度 (%)")

In [ ]:
# plot_importance — pc モジュールの関数として呼ぶ（結果は同じ）
ax = pc.plot_importance(result, color="#2CA02C", sort=False)
ax.set_title("pc.plot_importance() で描画（ソートなし）")

In [ ]:
# plot_partworth — result メソッドとして呼ぶ
ax = result.plot_partworth(title="部分効用（パートワース）", show_zero_line=True)

In [ ]:
# plot_partworth — pc モジュールの関数として呼ぶ
ax = pc.plot_partworth(result, show_zero_line=False)
ax.set_title("pc.plot_partworth()（ゼロ線なし）")

In [ ]:
# plot_wtp — result メソッドとして呼ぶ
ax = result.plot_wtp(price_unit="万円", show_values=True, sort=True)

In [ ]:
# plot_wtp — pc モジュールの関数として呼ぶ
ax = pc.plot_wtp(result, price_unit="万円", color="#9467BD", sort=False)
ax.set_title("pc.plot_wtp()（ソートなし）")